In [1]:
from __future__ import annotations

import os
import base64
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
from retreival import retrieval
from index_embedd import save,VectorStore
from data_gathering import ingest, Data, UserRequest
from LLM import  initialize_hf_llm
from index_embedd import initialize
import puremagic

from query_handler import handle_query


# Force Hugging Face to look directly at your D drive directory bypassing the link
os.environ["HF_HOME"] = r"D:\models\huggingface"
os.environ["TORCH_HOME"] = r"D:\models\torch_models"



In [3]:
load_dotenv()
hf_api_key = os.getenv("HUGGIN_FACE_API")

client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-8B-Instruct"

# Convert local image file to base64 string
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")




In [ ]:
##test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

# print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))



local_image_path = r"C:\Users\shahin\Desktop\pics\er.jfif"
base64_image = encode_image_to_base64(local_image_path)
image_data_url = f"data:image/jpeg;base64,{base64_image}"

# Request execution block
response = client.chat.completions.create(
    model=vision_model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this picture in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url  # Passes the parsed base64 data string variable
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [4]:

def detect_file_type(file_path):
    path = Path(file_path)

    suffix, mime = None, None

    if path.exists():
        suffix = path.suffix
        mime = puremagic.from_file(str(path), mime=True)

    return mime, suffix, path


#####alternatives:def robust_detect(path: str):
#    from pathlib import Path
# import mimetypes
# import filetype
#
# def robust_detect(path: str):
#     p = Path(path)
#
#     # 1. fast extension guess
#     mime, _ = mimetypes.guess_type(str(p))
#
#     # 2. fallback binary sniff
#     if mime is None:
#         kind = filetype.guess(str(p))
#         if kind:
#             mime = kind.mime
#
#     return {
#         "path": p,
#         "suffix": p.suffix,
#         "mime": mime
#     }

def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        print("word")
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:

        handle_pp(path)

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

In [13]:
from word_handler import process_docx


def handle_word(path: str):
    chunks, doc_meta, img_to_ch, tbl_to_ch = process_docx(path)

    if doc_meta.doc_id in VectorStore.indexed_docs:
        print(f"⏭️  skipped {doc_meta.doc_id} (already indexed)")
        return

    ingest(chunks, doc_meta, img_to_ch, tbl_to_ch)
    VectorStore.indexed_docs.add(doc_meta.doc_id)

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

In [ ]:
from your_router import classify_query, route
from your_pipeline import process_request
from dataclasses import dataclass



def test_case(query, images=None, docs=None):
    req = UserRequest(query, images, docs)

    result = process_request(req)

    print("\nQUERY:", query)
    print("RESPONSE:", result)


test_case("explain this pdf", docs=["paper.pdf"])
test_case("what is shown here", images=["img.png"])
test_case("find methodology section", docs=["paper.pdf"])

In [12]:
if __name__ == "__main__":
    initialize()
    initialize_hf_llm()

    query="آخرین بخش آزمایش"

    req = UserRequest(query = query)
    handle_query(req)


    # index_images(Data.chunks)
    # index_chunks(Data.chunks)
    # mime, suffix, path = detect_file_type(r"F:\university\az e riz\گزارش.docx")
    # route_file(mime, suffix, path)
    #
    # results = retrieval(query="آخرین بخش آزمایش")
    # print("final results")
    # current_direct = os.curdir
    # save(current_direct)
    # for r in results:
    #     print(r)
    #context = build_context(results, [])
    #print(context[:600])


word
⏭️  skipped گزارش.docx (already indexed)
⚠️  Low confidence — activating query expansion fallback...
   Expanded query: 'آخرین بخش آزمایش, بخش پایانی, بخش آخر, قسمت نهایی, مراحل نهایی, بخش دوم'
⚠️  Low image confidence — activating query expansion fallback...
[{'text': 'بخش دوم: روشن شدن شدن همه نمایشگر ها و شمارش از 2000 تا 0: ایده اصلی این است که در یک لحظه همزمان همه سون سگمت ها روشن باشند و یک عدد را نشان دهند ولی در هر لحظه با استفاده از portB فقط یکی فعال باشد و بتواند تغییر کند. سپس یک تاخیر بسیار بسیار کوچک که چشم ما نتواند آنرا تشخیص دهد ایجاد کرده ، آن سون سگمت را غیرفعال و سون سگمت سمت راستش را فعال کرده و به آن اجازه تغییر میدهیم. به این ترتیب به بیننده ایده تغییر همزمان سون سگمت ها را میدهیم. برای اینکار فقط کافی است حلقه while   را تغییر دهیم. بدین ترتیب که در یک حلقه از 2000 تا 0 ، ارقام یکان دهگان صدگان و هزارگان عدد را استخراج می\u200cکنیم.(با استفاده از تقسیم و باقی مانده گیری بر 10 ،100، 1000).  حال ابتدا 0x08 را در پورت B  میریزیم تا چپ ترین سون سگمت فعال شود. 